# Proximal Policy Optimization (PPO): the clipped-objective workhorse of deep RL

_Reinforcement Learning_

**Take the biggest policy-gradient step you safely can — a single clipped objective stops the update from moving the policy too far from the old one.**

Plain policy gradient (REINFORCE, mod-policy-gradient) has one dangerous flaw:
       it does not control how big a step it takes. A single update with a large learning rate or
       a large advantage can push the policy so far that it collapses &mdash; it forgets what it learned
       and may never recover, because the next batch of data is collected by the now-broken policy.

       TRPO (Trust Region Policy Optimization) fixed this by adding a hard constraint: keep the
       new policy within a trust region of the old one, measured by KL divergence. It works but is
       complicated &mdash; it needs a constrained second-order optimization.

---

This notebook is a practice scaffold for the **Proximal Policy Optimization (PPO): the clipped-objective workhorse of deep RL** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q gymnasium stable-baselines3
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — gymnasium + PyTorch / stable-baselines3 (Colab)

In [ ]:
# ============================================================
# TRACK A — the clean path: stable-baselines3 PPO on CartPole.
#   In Colab:  !pip install stable-baselines3 gymnasium torch
# ============================================================
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy

env = gym.make("CartPole-v1")

# A multilayer-perceptron policy; the defaults already encode the
# clip (clip_range=0.2), several epochs (n_epochs=10), and GAE (gae_lambda=0.95).
model = PPO(
    "MlpPolicy", env,
    clip_range=0.2,        # the epsilon of L^CLIP
    n_epochs=10,           # gradient epochs reused over each batch (safe via the clip)
    gae_lambda=0.95,       # GAE lambda: bias/variance of the advantage
    ent_coef=0.0,          # entropy bonus coefficient (c2)
    vf_coef=0.5,           # value-loss coefficient (c1)
    verbose=1,
)
model.learn(total_timesteps=50_000)   # CartPole is solved well before this
mean_r, std_r = evaluate_policy(model, env, n_eval_episodes=20)
print(f"mean reward over 20 episodes: {mean_r:.1f} +/- {std_r:.1f}")  # ~500 when solved


# ============================================================
# TRACK B — from-scratch PPO clipped loss in PyTorch (the core).
#   This is the exact L^CLIP + value loss + entropy bonus the
#   lesson derives. Assumes you already collected a rollout batch.
# ============================================================
import torch
import torch.nn as nn

EPS, C1, C2 = 0.2, 0.5, 0.01   # clip epsilon, value coef, entropy coef

def ppo_loss(dist, value, actions, old_logp, advantages, returns):
    """One PPO loss on a minibatch.
       dist        : current policy's action distribution (torch.distributions)
       value       : critic V(s) for each state, shape [N]
       actions     : actions taken under the OLD policy, shape [N]
       old_logp    : log pi_old(a|s) recorded at collection time, shape [N]
       advantages  : GAE advantages \hat A_t, shape [N] (normalize before!)
       returns     : value targets \hat V_t for the critic, shape [N]
    """
    # --- the probability ratio r_t = pi_theta / pi_theta_old ---
    new_logp = dist.log_prob(actions)
    ratio = torch.exp(new_logp - old_logp)          # exp(log r) is numerically safer

    # --- the clipped surrogate: min(r*A, clip(r,1-eps,1+eps)*A) ---
    unclipped = ratio * advantages
    clipped   = torch.clamp(ratio, 1.0 - EPS, 1.0 + EPS) * advantages
    policy_loss = -torch.min(unclipped, clipped).mean()   # minimize -L^CLIP == maximize L^CLIP

    # --- value-function loss: keep the critic that feeds A_t accurate ---
    value_loss = (returns - value).pow(2).mean()

    # --- entropy bonus: reward an uncertain policy to preserve exploration ---
    entropy = dist.entropy().mean()

    # total: maximize L^CLIP, minimize value error, maximize entropy
    return policy_loss + C1 * value_loss - C2 * entropy

# Typical training loop (sketch):
#   for update in range(num_updates):
#       batch = collect_rollouts(policy, env)          # ON-POLICY, with old_logp recorded
#       adv, ret = compute_gae(batch, gamma=0.99, lam=0.95)
#       adv = (adv - adv.mean()) / (adv.std() + 1e-8)  # normalize advantages
#       for epoch in range(10):                        # reuse the batch — safe via the clip
#           for mb in minibatches(batch):
#               dist, value = policy(mb.states)
#               loss = ppo_loss(dist, value, mb.actions, mb.old_logp, adv[mb.idx], ret[mb.idx])
#               opt.zero_grad(); loss.backward()
#               nn.utils.clip_grad_norm_(policy.parameters(), 0.5)  # grad clip, separate from L^CLIP
#               opt.step()

## Visualize the data & results

_What does the PPO clipped objective actually look like as a function of the probability ratio r — and how does it differ from the unclipped surrogate? We compute min(r*A, clip(r,1-eps,1+eps)*A) and the raw r*A over a range of r in numpy, for a GOOD action (A=+1) and a BAD action (A=-1), and plot them. The clip should flatten the objective outside 1 +/- eps._

In [ ]:
import numpy as np

# The PPO per-sample clipped surrogate, as a function of the ratio r.
eps = 0.2                                  # clip epsilon -> clip range [0.8, 1.2]
r = np.linspace(0.0, 2.0, 11)              # probability ratios to evaluate

def surrogate(r, A):
    unclipped = r * A
    clipped   = np.clip(r, 1 - eps, 1 + eps) * A
    return np.minimum(unclipped, clipped)  # PPO's L^CLIP per sample

A_pos, A_neg = 1.0, -1.0
print("r            :", np.round(r, 2))
print("unclipped A=+1:", np.round(r * A_pos, 2))      # the gray line: keeps rising
print("clipped   A=+1:", np.round(surrogate(r, A_pos), 2))  # flattens at 1.2 past r=1.2
print("clipped   A=-1:", np.round(surrogate(r, A_neg), 2))  # flat left of r=0.8

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** With $\epsilon = 0.2$, an advantage $A_t = +3$, and a ratio $r_t = 1.4$, what value does the PPO clipped objective take for this sample, and what is its gradient with respect to $r_t$ there?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compute the unclipped term. — _$r_t A_t = 1.4 \times 3 = 4.2$._
- Compute the clipped term. — _$\text{clip}(1.4, 0.8, 1.2) = 1.2$, so $1.2 \times 3 = 3.6$._
- Take the min. — _$\min(4.2, 3.6) = 3.6$ — the clipped value, because $A_t \gt 0$ and $r_t$ exceeded $1+\epsilon$._

**Answer:** The objective is $3.6$, and because the clipped term (flat in $r_t$) is selected, the gradient with respect to $r_t$ is $0$. PPO gives no incentive to push this already-favored good action any higher in this update.

</details>

**Problem 2.** Why does PPO use $\min(r_t A_t,\ \text{clip}(r_t,1-\epsilon,1+\epsilon)A_t)$ instead of just the clipped term $\text{clip}(r_t,1-\epsilon,1+\epsilon)A_t$?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Consider a bad action whose ratio already grew past $1+\epsilon$. — _Say $A_t \lt 0$ and $r_t = 1.5$ — we accidentally made a harmful action more likely._
- Check the clipped-only objective there. — _$\text{clip}(1.5,\dots)$ freezes at $1+\epsilon$, so the term is flat in $r_t$ — zero gradient, no way to correct._
- Check the min. — _The unclipped $r_t A_t$ is more negative, so the $\min$ selects it, restoring a gradient that lowers the bad action's probability._

**Answer:** The $\min$ keeps the objective pessimistic: when an action has already over-shot in a harmful direction, it reactivates the unclipped term so the policy can pull back. The clip removes the incentive to overshoot; the $\min$ guarantees a past overshoot is still correctable.

</details>

**Problem 3.** PPO runs several gradient epochs over the SAME batch of data. Why is that safe here when it would be dangerous for plain REINFORCE, and what failure can still occur if you use too many epochs?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall why repeated updates on one batch are risky. — _Each epoch moves $\theta$ further from $\theta_{old}$; on-policy data becomes stale and the policy can over-fit that one batch._
- See what the clip does. — _The clip flattens the objective once any sample's ratio leaves $1\pm\epsilon$, removing the gradient that would push it further — bounding per-sample drift._
- Note the residual risk. — _The clip bounds each sample, not the total KL divergence; enough epochs can still accumulate large overall drift._

**Answer:** The clip caps how much any single action's probability can be rewarded for moving, so reusing the batch a few times stays within a soft trust region instead of collapsing the policy. But the clip does not bound total KL divergence: too many epochs can still let the new policy drift far from the old one, so practitioners cap epochs (3&ndash;10) and early-stop on a KL target.

</details>